# Notes

code for producing GOIT pipelines summary stats, and for calculating landing page stats

this is saved as an Excel file, which Baird copies/pastes into the existing summary tables information on the drive here:
https://docs.google.com/spreadsheets/d/1OYH6D7c-D0FsL5GzBGijtkmvQCTkBUclj-UVoOieUFo/edit

In [567]:
import pandas
import numpy
import pygsheets
import datetime
import re
import pytz

In [568]:
# define the excel file to save tables in
current_time = datetime.datetime.now(pytz.timezone('US/Eastern')).strftime("%Y-%m-%d_T%H%M%S")

In [ ]:
gas_fuel_options = ['Gas', 'Gas and Hydrogen']
gas_hydrogen_fuel_options = ['Gas',
                             'Hydrogen'
                            ]
hydrogen_fuel_options = [
                             'Hydrogen'
                            ]

# oil + NGL fuel buckets — union must equal the 'Oil-NGL' fuel_options list in
# GOIT-GGIT-data-requests/notebooks/convert-ggit-goit-to-tracker-release-downloads-CURRENT.ipynb.
# 'Oil, NGL' and 'Oil, NGL, naphtha' intentionally appear in BOTH buckets.

oil_fuel_options = ['Oil',
                    'Oil, NGL',
                    'Oil, NGL, naphtha',
                    'Oil products (only)',
                    'Oil, oil products',
                    'Oil, condensate',
                    'CO2',
                   ]

ngl_fuel_options = ['NGL',
                    'NGL, oil products',
                    'Oil, NGL',
                    'Oil, NGL, naphtha',
                    'LPG',
                    'Naphtha (only)',
                    'Naphtha, oil products',
                    'Condensate',
                    'Condensate/NGL',
                   ]

In [ ]:
#fuel_type = 'Gas'
fuel_type = 'Oil'
#fuel_type = 'NGL'

## import data

In [ ]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA') # file to use for gas pipelines Dec 2023
# spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # CURRENT sheet
#spreadsheet = gc.open_by_key('1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs') # dec 2024 dataset for GGIT release
#spreadsheet = gc.open_by_key('1xjaeq0OwdN-Orht7Q7ynPHB2uY_gnMvAeha-QHkzoZw') # Nov 2025 for GGIT
spreadsheet = gc.open_by_key('1DrG5_zSa-PxLBaa1eBAOgONISgjeM7Te1El6SObiq3E') # June 2026 GOIT oil/NGL release

gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
oil_pipes = oil_pipes.drop('WKTFormat', axis=1)

#get other relevant sheets
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
owners_df_orig = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')

country_ratios_df = country_ratios_df.loc[country_ratios_df.Wiki!='']

# pick the source dataframe based on fuel_type
if fuel_type == 'Gas':
    pipes_df_orig = gas_pipes.copy()
if fuel_type in ['Oil', 'NGL']:
    pipes_df_orig = oil_pipes.copy()

# remove empty cells for pipes, owners
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Wiki']!='']
# subset for fuel type
if fuel_type == 'Gas':
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(gas_fuel_options)]
    hydro_mask = pipes_df_orig.Fuel == 'Gas and Hydrogen'
    pipes_df_orig.loc[hydro_mask, 'Fuel'] = 'Gas'
    # do same for country_ratios
    hydro_mask = country_ratios_df.Fuel == 'Gas and Hydrogen'
    country_ratios_df.loc[hydro_mask, 'Fuel'] = 'Gas'
elif fuel_type == 'Oil':
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(oil_fuel_options)]
elif fuel_type == 'NGL':
    pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(ngl_fuel_options)]

owners_df_orig = owners_df_orig.loc[owners_df_orig['ProjectID']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig['Wiki']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig.Status!='N/A']

owners_df_orig.set_index('ProjectID', inplace=True)

parent_metadata_df = spreadsheet.worksheet('title', 'Parent metadata (3/3)').get_as_df(start='A3')
parent_metadata_df.set_index('Parent', inplace=True)

In [572]:
country_ratios_df.replace('--', numpy.nan, inplace=True)

owners_df_orig.replace('',numpy.nan,inplace=True)
owners_df_orig.replace('--',numpy.nan,inplace=True)

pipes_df_orig.replace('--',numpy.nan,inplace=True)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_8949/1702877721.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df.replace('--', numpy.nan, inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_8949/1702877721.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  owners_df_orig.replace('',numpy.nan,inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_8949/1702877721.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future

In [ ]:
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

region_name = 'Global'; region_df_touse = region_df_orig.copy()
#region_name = 'AsiaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AsiaGasTracker=='Yes']
#region_name = 'EuroGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.EuroGasTracker=='Yes']
#region_name = 'AfricaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AfricaGasTracker=='Yes']
#region_name = 'LatinAmericaTracker'; region_df_touse = region_df_orig.loc[region_df_orig.LatinAmericaTracker=='Yes']

In [596]:
region_df_touse_cleaned = region_df_touse.loc[(region_df_touse.Region!='--')&
                                            (region_df_touse.SubRegion!='--')]
multiindex_region_subregion = region_df_touse_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion

MultiIndex([('Americas', 'Latin America and the Caribbean')],
           names=['Region', 'SubRegion'])

## file names with regional specifics

In [ ]:
if fuel_type=='Gas':
    excel_writer = pandas.ExcelWriter('GOIT-Summary-Sheets-Gas-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='NGL':
    excel_writer = pandas.ExcelWriter('GOIT-Summary-Sheets-NGL-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='Oil':
    excel_writer = pandas.ExcelWriter('GOIT-Summary-Sheets-Oil-'+str(datetime.date.today())+'.xlsx')

### create country-specific dataframes for region, country_ratios_df, owners_df

In [598]:
country_ratios_df_touse = country_ratios_df.loc[country_ratios_df['Country'].str.contains(
                                            '|'.join(region_df_touse['Country'].tolist()))]

# owners_df_touse = owners_df_orig.loc[owners_df_orig['Countries'].str.contains(
#                                             '|'.join(region_df_touse['Country'].tolist()))]

pipes_df_touse = pipes_df_orig.loc[pipes_df_orig['CountriesOrAreas'].str.contains(
                                            '|'.join(region_df_touse['Country'].tolist()))]

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_8949/3448576929.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  country_ratios_df_touse = country_ratios_df.loc[country_ratios_df['Country'].str.contains(
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_8949/3448576929.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pipes_df_touse = pipes_df_orig.loc[pipes_df_orig['CountriesOrAreas'].str.contains(


In [599]:
country_ratios_df

,PipelineName,SegmentName,ProjectID,Country,LengthEstimateKmByCountry,LengthPerCountryFraction,Region,SubRegion,Wiki,Status,...,H2Type,CancelledYear,ProposalYear,ConstructionYear,ShelvedYear,StartYearEarliest,StartCountry,EndCountry,RouteType,RouteAccuracy
0,Alberta Clipper Oil Pipeline,,P0001,Canada,1066.328784,0.680000,Americas,Northern America,https://www.gem.wiki/Alberta_Clipper_Oil_Pipeline,operating,...,NaN,,,,,2010.0,Canada,United States,Mapped route (at any accuracy),high
1,Alberta Clipper Oil Pipeline,,P0001,United States,512.042577,0.320000,Americas,Northern America,https://www.gem.wiki/Alberta_Clipper_Oil_Pipeline,operating,...,NaN,,,,,2010.0,Canada,United States,Mapped route (at any accuracy),high
2,Athabasca Oil Pipeline,,P0002,Canada,522.239984,1.000000,Americas,Northern America,https://www.gem.wiki/Athabasca_Oil_Pipeline,operating,...,NaN,,,,,1999.0,Canada,Canada,Mapped route (at any accuracy),high
3,Bakken Expansion Pipeline,,P0004,Canada,150.276903,0.595268,Americas,Northern America,https://www.gem.wiki/Bakken_Expansion_Pipeline,operating,...,NaN,,,,,2013.0,United States,Canada,Mapped route (at any accuracy),high
4,Bakken Expansion Pipeline,,P0004,United States,102.1756,0.400000,Americas,Northern America,https://www.gem.wiki/Bakken_Expansion_Pipeline,operating,...,NaN,,,,,2013.0,United States,Canada,Mapped route (at any accuracy),high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6720,Spring Creek Gas Infrastructure Expansion Project,Backbone,P7823,United States,19.452577,1.000000,Americas,Northern America,https://www.gem.wiki/Spring_Creek_Gas_Infrastr...,construction,...,,,2019,2020,,2027.0,United States,United States,Mapped route (at any accuracy),high
6721,National Fuel Gas Supply Corporation Gas Pipeline,Tioga Pathway Expansion,P7827,United States,38.204595,1.000000,Americas,Northern America,https://www.gem.wiki/National_Fuel_Gas_Supply_...,proposed,...,,,,2025,,2026.0,United States,United States,Mapped route (at any accuracy),high
6723,Pelican Pipeline,,P7831,United States,246.66962,1.000000,Americas,Northern America,https://www.gem.wiki/Pelican_Pipeline,construction,...,,,2024,2025,,2027.0,United States,United States,Mapped route (at any accuracy),high
6731,Transwestern Gas Pipeline,Desert Southwest Expansion,P7840,United States,813.705968,1.000000,Americas,Northern America,https://www.gem.wiki/Transwestern_Gas_Pipeline,proposed,...,,,2025,,,2029.0,United States,United States,Mapped route (at any accuracy),high


In [600]:
pipes_df_touse.head()

,PipelineNetworkGrouping,PipelineNetworkGroupingAlt,PipelineName,Wiki,SegmentName,OtherEnglishNames,ProjectID,Status,Status [ref],Researcher,...,PipelineDirectionality,SciGridNames,QCCOwner,QCCOwner [ref],ImpactedByRussiaUkraineInvasion,EGTImport,ChinesePipelineType,ChineseNetworkPrimary,ChineseNetworkSecondary,ChineseClassificationNotes
69,,,Mier Monterrey Gas Pipeline,https://www.gem.wiki/Mier_Monterrey_Gas_Pipeline,,,P0215,operating,,GC,...,,,,,,,,,,
106,Sur de Texas-Coatzacoalcos Pipeline Network,,Sur de Texas-Tuxpan Gas Pipeline,https://www.gem.wiki/Sur_de_Texas-Tuxpan_Gas_P...,,,P0257,operating,,GC,...,,,,,,,,,,
139,,,Nueva Era Pipeline,https://www.gem.wiki/Nueva_Era_Pipeline,,"Impulsora Pipeline, Gasoducto Nueva Era",P0290,operating,,GC,...,,,,,,,,,,
176,,,Energia Mayakan Pipeline,https://www.gem.wiki/Energia_Mayakan_Pipeline,,,P0339,operating,,GC,...,,,,,,,,,,
190,,,Paso Norte Pipeline,https://www.gem.wiki/Paso_Norte_Pipeline,,,P0382,cancelled,https://www.pgjonline.com/media/9572/pn_0123_o...,GC,...,,,,,,,,,,


### sum LengthMergedKmByCountry and MergedKmByRegion

In [601]:
status_list = ['proposed', 
               'construction', 
               'shelved', 
               'cancelled', 
               'operating', 
               'idle', 
               'mothballed', 
               'retired']
country_list = sorted(list(set(country_ratios_df_touse['Country'])))
region_list = sorted(list(set(country_ratios_df_touse['Region'])))

In [602]:
excel_status_list = ['proposed', 
                     'construction', 
                     'in development (proposed + construction)', 
                     'shelved', 
                     'cancelled', 
                     'operating', 
                     'idle', 
                     'mothballed', 
                     'retired']
excel_status_list_with_countries = ['Country']+excel_status_list

In [ ]:
if fuel_type == 'Gas':
    fuel_options = gas_fuel_options
elif fuel_type == 'Oil':
    fuel_options = oil_fuel_options
elif fuel_type == 'NGL':
    fuel_options = ngl_fuel_options

country_ratios_df_subset = country_ratios_df_touse.loc[country_ratios_df_touse['Fuel'].isin(fuel_options)]

km_by_country = pandas.DataFrame(columns=status_list, index=country_list)
km_by_region = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_country[status] = country_ratios_df_subset_status.groupby('Country')['LengthMergedKmByCountry'].sum()

print('===regional calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_region[status] = country_ratios_df_subset_status.groupby(['Region','SubRegion'])['LengthMergedKmByCountry'].sum()

# fille NaN with 0.0
km_by_region = km_by_region.fillna(0)
km_by_country = km_by_country.fillna(0)

km_by_region['in development (proposed + construction)'] = km_by_region[['proposed','construction']].sum(axis=1)
km_by_country['in development (proposed + construction)'] = km_by_country[['proposed','construction']].sum(axis=1)

km_by_country = km_by_country[excel_status_list]
km_by_region = km_by_region[excel_status_list]

km_by_region.index.names = ['Region','Subregion']
km_by_country.index.name = 'Country'

km_by_region.loc['Total',:] = km_by_region.sum(axis=0).values
km_by_country.loc['Total',:] = km_by_country.sum(axis=0).values

# drop all-zero rows
km_by_country = km_by_country.loc[~(km_by_country==0).all(axis=1)]

km_by_country.replace(0,'',inplace=True)
km_by_region.replace(0,'',inplace=True)

km_by_region.to_excel(excel_writer, 'Kilometers by region')
km_by_country.to_excel(excel_writer, 'Kilometers by country')

In [604]:
km_by_region

,,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired
Region,Subregion,,,,,,,,,
Americas,Latin America and the Caribbean,20388.29,1311.8,21700.09,4337.0,5020.66,69076.06,331.0,224.0,
Total,,20388.29,1311.8,21700.09,4337.0,5020.66,69076.06,331.0,224.0,


## pipeline km by parent company (owner) and project status

### first check that there are no missing projectids

In [605]:
import pandas
import numpy
import re

# First check that there are no missing ProjectID values
if country_ratios_df_subset['ProjectID'].isna().any():
    missing_rows = country_ratios_df_subset[country_ratios_df_subset['ProjectID'].isna()]
    raise ValueError(f"Missing ProjectID in these rows: {missing_rows.index.tolist()}")

owner_parent_calculations_df = pandas.DataFrame()
# needs country, km in each country columns as well

for idx, row in country_ratios_df_subset.iterrows():
    # print(row.ProjectID)
    parent_string = row.Parent
    # print(parent_string)

    # if the first letter of the parent_string is a chinese character, and it ends with [100.00%],
    # that means it's a non-researched QCC owner; keep as-is
    if re.findall(r'[\u4e00-\u9fff]+', parent_string[:1]) != [] and parent_string[-9:] == '[100.00%]':
        parent_list = [parent_string.split(' [100.00%]')[0]]
        percent_list = [1.0]
    # otherwise do the rest of the thing
    else:
        # ✅ make this a raw string so '\[' is not treated as an escape
        parent_list = re.sub(r' \[.*?\]', '', parent_string).split('; ')  # all entries must have a Owner [%] syntax
        # ✅ cleaner as raw string too
        percent_list = [
            float(i.rstrip('%')) / 100.
            for i in re.findall(r'\d+(?:\.\d+)?%', parent_string)
        ]

    if parent_list.__len__() != percent_list.__len__():
        # print(parent_list)
        if percent_list == []:
            percent_list = [1 / parent_list.__len__() for i in parent_list]
        else:
            nmissing = parent_list.__len__() - percent_list.__len__()
            # distribute nans evenly
            total = numpy.nansum(percent_list)
            leftover = 1 - total
            percent_list += [leftover / nmissing] * nmissing

    for p_idx, parent in enumerate(parent_list):
        owner_parent_calculations_df = pandas.concat([
            owner_parent_calculations_df,
            pandas.DataFrame([{
                'Parent': parent,
                'ProjectID': row.ProjectID,
                'FractionOwnership': percent_list[p_idx],
                # 'ParentHQCountry': parent_metadata_df.loc[parent,'ParentHQCountry'],
                # 'ParentHQRegion': parent_metadata_df.loc[parent,'ParentHQRegion'],
                'Country': row.Country,
                'Status': row.Status,
                'LengthMergedKmByCountry': row.LengthMergedKmByCountry
            }])
        ])

owner_parent_calculations_df['KmOwnership'] = (
    owner_parent_calculations_df.FractionOwnership *
    owner_parent_calculations_df.LengthMergedKmByCountry
)

In [606]:
owner_parent_calculations_df

,Parent,ProjectID,FractionOwnership,Country,Status,LengthMergedKmByCountry,KmOwnership
0,Kinder Morgan Inc,P0184,1.0,Mexico,operating,0.14,0.140
0,Kinder Morgan Inc,P0215,1.0,Mexico,operating,144.81,144.810
0,TC Energy Corp,P0257,0.6,Mexico,operating,758.53,455.118
0,Sempra Energy,P0257,0.4,Mexico,operating,758.53,303.412
0,Kinder Morgan Inc,P0258,1.0,Mexico,operating,8.07,8.070
...,...,...,...,...,...,...,...
0,unknown,P7816,1.0,Trinidad and Tobago,proposed,62.29,62.290
0,unknown,P7816,1.0,Guyana,proposed,428.17,428.170
0,unknown,P7816,1.0,Venezuela,proposed,158.15,158.150
0,unknown,P7816,1.0,Suriname,proposed,51.39,51.390


In [607]:
excel_status_list_with_countries

['Country',
 'proposed',
 'construction',
 'in development (proposed + construction)',
 'shelved',
 'cancelled',
 'operating',
 'idle',
 'mothballed',
 'retired']

In [608]:
#unique_owner_list = owner_parent_calculations_df.Parent.sort_values().unique().tolist()

##################################################
# create km count by owner, status
##################################################
owners_km_by_status_df = \
    owner_parent_calculations_df.groupby(
    ['Parent',
     # 'ParentHQCountry',
     'Status'])[['KmOwnership']].sum().unstack().droplevel(axis=1, level=[0])

owners_km_by_status_df = owners_km_by_status_df.reindex(columns=status_list)
owners_km_by_status_df = owners_km_by_status_df.reset_index().set_index('Parent')
owners_km_by_status_df.columns.name = None

owners_km_by_status_df['in development (proposed + construction)'] = owners_km_by_status_df[['proposed','construction']].sum(axis=1)

owners_km_by_status_df = owners_km_by_status_df.rename(columns={'Parent':'Parent Company',
                                                                          # 'ParentHQCountry':'Country'
                                                               })
# rearrange the order of the columns for output
owners_km_by_status_df = owners_km_by_status_df[excel_status_list]#_with_countries]

# totals_row = owners_ntrains_by_status_df.sum(axis=0, min_count=0)
# totals_row.name = 'Total'
# owners_ntrains_by_status_df = owners_ntrains_by_status_df.append(totals_row)
owners_km_by_status_df.loc['Total',:] = owners_km_by_status_df.sum(axis=0, min_count=0).values
owners_km_by_status_df.loc['Total','Country'] = ''

owners_km_by_status_df = owners_km_by_status_df.replace(numpy.nan, '')
owners_km_by_status_df = owners_km_by_status_df.replace(0, '')

owners_km_by_status_df.to_excel(excel_writer, sheet_name='Kilometers by owner')

In [609]:
owners_km_by_status_df

,proposed,construction,in development (proposed + construction),shelved,cancelled,operating,idle,mothballed,retired,Country
Parent,,,,,,,,,,
1832 Asset Management LP,,,,,,20.7292,,,,
AES Corp,,,,,,94.29,,,,
AVAIO MPL Special LP,200.0,,200.0,,,,,,,
Administracion Nacional de Combustibles Alcohol y Portland,,,,,,107.2,,,,
Alberta Investment Management Corp,,,,,,127.99875,,,,
...,...,...,...,...,...,...,...,...,...,...
Wärtsilä Oyj,,,,,30.666667,,,,,
Yacimientos Petrolíferos Fiscales Bolivianos Corp,,,,1350.0,,2048.15608,,,,
small shareholder(s),,,,,,8.12392,,,,


### pipeline km by start year, type

In [ ]:
pipes_started = pipes_df_touse.copy()
#pipes_started['StartYearLatest'].replace(numpy.nan,'',inplace=True)

pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel'].isin(fuel_options))]

pipes_started_sum = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()

In [611]:
pipes_started_sum

StartYearEarliest
1950.0     2470.00
1951.0      772.00
1960.0     1454.00
1965.0     4590.00
1969.0      357.00
1970.0     2046.00
1972.0      628.28
1974.0      225.00
1977.0     2537.00
1981.0     1401.00
1984.0      340.00
1986.0     1366.80
1988.0     3139.00
1994.0      133.00
1995.0      485.00
1996.0     1996.00
1997.0     1539.70
1998.0      846.20
1999.0    15443.00
2000.0     1297.80
2001.0      311.00
2002.0     1262.80
2003.0      817.84
2004.0     1410.00
2005.0       76.50
2006.0     1253.30
2007.0      499.90
2008.0     1086.94
2009.0      841.60
2010.0     1408.25
2011.0      724.00
2012.0       54.80
2013.0      384.46
2014.0     2304.56
2016.0     1840.00
2017.0      282.00
2018.0     1418.87
2019.0     2014.00
2020.0      621.00
2021.0      650.00
2022.0      320.00
2023.0      815.40
2024.0      993.60
2025.0      715.00
Name: LengthMergedKm, dtype: float64

In [ ]:
pipes_proposed = pipes_df_touse.copy()

pipes_proposed = pipes_proposed[(pipes_proposed['Status'].isin(['proposed'])) &
                                (pipes_proposed['Fuel'].isin(fuel_options))]

pipes_proposed_sum = pipes_proposed.groupby('ProposalYear')['LengthMergedKm'].sum()

In [ ]:
pipes_construction = pipes_df_touse.copy()

pipes_construction = pipes_construction[(pipes_construction['Status'].isin(['construction'])) &
                                        (pipes_construction['Fuel'].isin(fuel_options))]

pipes_construction_sum = pipes_construction.groupby('ConstructionYear')['LengthMergedKm'].sum()

In [ ]:
if fuel_type == 'Gas':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)))
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Gas pipeline km operating'] = pipes_started_sum
    km_by_start_year['Gas pipeline km construction'] = pipes_construction_sum
    km_by_start_year['Gas pipeline km proposed'] = pipes_proposed_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'Oil':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)))
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Oil pipeline km operating'] = pipes_started_sum
    km_by_start_year['Oil pipeline km construction'] = pipes_construction_sum
    km_by_start_year['Oil pipeline km proposed'] = pipes_proposed_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'NGL':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)))
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['NGL pipeline km operating'] = pipes_started_sum
    km_by_start_year['NGL pipeline km construction'] = pipes_construction_sum
    km_by_start_year['NGL pipeline km proposed'] = pipes_proposed_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

km_by_start_year.loc['Total',:] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')
#km_by_start_year

## cost calculations

In [ ]:
temp_df = pipes_df_orig.loc[(~pipes_df_orig.CostUSDPerKm.isnull())&
                            (pipes_df_orig.Fuel.isin(fuel_options))]
qlo_val = 0.025
qhi_val = 0.975

q_lo=temp_df['CostUSDPerKm'].quantile(qlo_val)
q_hi=temp_df['CostUSDPerKm'].quantile(qhi_val)
print(temp_df['CostUSDPerKm'].quantile(qlo_val))
print(temp_df['CostUSDPerKm'].quantile(qhi_val))

temp_df = temp_df.loc[temp_df['CostUSDPerKm'].between(q_lo, q_hi, inclusive='neither')]

## save excel file

In [614]:
excel_writer.close()

## calculating stats for landing page

In [ ]:
# number of projects tracked in total
print(pipes_df_touse.loc[pipes_df_touse.Fuel.isin(fuel_options)].shape[0], fuel_type, 'pipeline projects tracked')
print(pipes_df_touse.loc[pipes_df_touse.Fuel.isin(fuel_options)]['LengthMergedKm'].sum()/1e6, 'M km tracked')

In [ ]:
# number of projects tracked in total
print(pipes_df_touse.loc[(pipes_df_touse.Fuel.isin(fuel_options))&
                         (pipes_df_touse.Status.isin(['proposed','construction']))].shape[0], fuel_type, 'pipeline projects tracked')
print(pipes_df_touse.loc[(pipes_df_touse.Fuel.isin(fuel_options))&
                         (pipes_df_touse.Status.isin(['proposed','construction']))]['LengthMergedKm'].sum()/1e3, 'K km tracked')